# Stuttering Detection: Neural Network (Deep Learning) Analysis
**Course**: CS204T (Artificial Intelligence)  
**Team**: 18  
**Focus**: Shallow Architecture vs. Deep Multi-Layer Perceptrons

---

## Step 1: Initialization & PyTorch Environment
We use the PyTorch framework. We have enabled **IPython Autoreload** to ensure any changes to our source code modules in `src/` are automatically detected by the notebook.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import shutil
import numpy as np
import torch.nn as nn
import torch.optim as optim
from src.extractors import WavLMExtractor
from src.data import DataManager
from src.models import ShallowNeuralNetwork, DeepNeuralNetwork

# Dataset Configuration
AUDIO_DIR = "Stuttering Events in Podcasts Dataset/clips/stuttering-clips/clips"
CSV_PATHS = [
    "Stuttering Events in Podcasts Dataset/SEP-28k_labels.csv",
    "Stuttering Events in Podcasts Dataset/fluencybank_labels.csv"
]
FEATURE_DIR = "data/features"
fluent_dir = os.path.join(FEATURE_DIR, "fluent")
disfluent_dir = os.path.join(FEATURE_DIR, "disfluent")

## Step 2 (Optional): Operational Mode for Data Extraction
Toggle how you want to handle your audio data for this session.
* `SKIP_EXTRACTION`: Uses features already on disk (Default).
* `FORCE_EXTRACT`: Analyzes raw audio for new files (Resumable).
* `CLEAN_START`: Wipes the database and re-extracts from zero.

In [ ]:
# Operational Flags
SKIP_EXTRACTION = True
FORCE_EXTRACT = False
CLEAN_START = False
NUM_CLIPS_TO_EXTRACT = 1000

if CLEAN_START:
    if os.path.exists(FEATURE_DIR):
        shutil.rmtree(FEATURE_DIR)
    print("[System] Clean start initiated. Feature database wiped.")

if not SKIP_EXTRACTION or CLEAN_START or FORCE_EXTRACT:
    extractor = WavLMExtractor("microsoft/wavlm-base")
    label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True)
    
    # Native randomized sampling of the dataset
    extractor.extract_from_dir(
        AUDIO_DIR, 
        output_dir=FEATURE_DIR, 
        label_dict=label_dict, 
        limit=NUM_CLIPS_TO_EXTRACT, 
        random_sample=True
    )
else:
    print("[System] Skipping extraction. Using existing data on disk.")

## Step 3: Data Preparation Engine
We load our distributed features, perform **Class Balancing** via oversampling, and standardize the features. Standard scaling is critical for Neural Networks to ensure the gradients don't explode during training.

In [ ]:
label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True)
manager = DataManager(None, None)

# Loading .npy features
X, y = manager.load_from_folders(fluent_dir, disfluent_dir)
X_train, X_val, X_test, y_train, y_val, y_test = manager.get_splits(test_size=0.15, val_size=0.15)

# Balance Training Set (50/50 split)
X_train_bal, y_train_bal = manager.balance_data(X_train, y_train, strategy="oversample")

# Anti-Leakage Standard Selection
X_train_final = manager.preprocess(X_train_bal, method="standard", fit=True)
X_test_final = manager.preprocess(X_test, method="standard", fit=False)

## Step 4: Model 1 - Shallow Neural Network
Our shallow architecture uses a single hidden layer of 64 neurons. 

**Explicit Configuration**: 
* `activation_fn`: Sigmoid (Classical probabilistic non-linearity)
* `optimizer_class`: SGD (Stochastic Gradient Descent)
* `lr`: Learning Rate (Controls convergence speed)

In [ ]:
# --- [Hyperparameter Panel] ---
# Activation: nn.Sigmoid, nn.ReLU, nn.Tanh, nn.LeakyReLU
# Optimizer:  optim.SGD, optim.Adam, optim.Adagrad
# LR values:   0.01 (Standard), 0.001 (Slow/Stable), 0.1 (Aggressive)

shallow_nn = ShallowNeuralNetwork(
    model_name="Shallow_NN_64", 
    hidden_layer_size=64, 
    epochs=200,
    lr=0.01,
    activation_fn=nn.Sigmoid,
    optimizer_class=optim.SGD
)

shallow_nn.train(X_train_final, y_train_bal)

print("\n--- Evaluation on Unseen Test Set ---")
shallow_nn.evaluate(X_test_final, y_test)

## Step 5: Model 2 - Deep Neural Network
To capture more complex relationships, we add a second hidden layer. The architecture [128, 64] allows the model to learn abstract features before making the final binary classification.

In [ ]:
deep_nn = DeepNeuralNetwork(
    model_name="Deep_NN_128_64", 
    hidden_layer_sizes=[128, 64], 
    epochs=200,
    lr=0.01,
    activation_fn=nn.Sigmoid,    
    optimizer_class=optim.SGD     
)

deep_nn.train(X_train_final, y_train_bal)

print("\n--- Evaluation on Unseen Test Set ---")
deep_nn.evaluate(X_test_final, y_test)